In [1]:
import sys
sys.path.append("../")

import pandas as pd
from app.services.resume_parser import (
    get_resume_text,
    extract_text_from_pdf,
    extract_text_from_docx,
    extract_name,
    extract_email,
    extract_phone,
    extract_entities,
)
import os

df = pd.read_csv("../data/Resume.csv")
print(df.shape)
df.head()

(2484, 4)


,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [2]:
def safe_get_text(row=None, file_path=None):
    """
    Tries file-based extraction first (if file_path given and exists),
    otherwise falls back to the CSV's Resume_str column.
    Never raises — returns "" on total failure.
    """
    if file_path and os.path.exists(file_path):
        ext = os.path.splitext(file_path)[1].lower()
        try:
            with open(file_path, "rb") as f:
                file_bytes = f.read()
            text = get_resume_text(file_bytes=file_bytes, file_extension=ext)
            if text:
                return text
        except Exception:
            pass  # fall through to CSV text below

    if row is not None:
        return get_resume_text(raw_text=row.get("Resume_str", ""))

    return ""

In [3]:
sample_df = df.sample(n=20, random_state=42).copy()

results = []
for _, row in sample_df.iterrows():
    text = safe_get_text(row=row, file_path=None)
    results.append({
        "ID": row["ID"],
        "category": row["Category"],
        "text_length": len(text),
        "name": extract_name(text),
        "email": extract_email(text),
        "phone": extract_phone(text),
        "num_entities": len(extract_entities(text)),
    })

results_df = pd.DataFrame(results)
results_df

,ID,category,text_length,name,email,phone,num_entities
0,99244405,TEACHER,5557,NaN,None,None,10
1,17562754,DIGITAL-MEDIA,6438,NaN,None,None,30
2,30311725,CONSTRUCTION,5699,NaN,None,None,13
3,19007667,CHEF,1471,NaN,None,None,10
4,11065180,BANKING,4958,NaN,None,None,30
5,39237915,BUSINESS-DEVELOPMENT,5629,NaN,None,None,30
6,17199951,DESIGNER,3793,Job Captain,None,None,30
7,18236085,BUSINESS-DEVELOPMENT,5524,NaN,None,None,30
8,79663360,TEACHER,5312,NaN,None,None,19
9,62312955,DESIGNER,2739,NaN,None,None,29


In [4]:
# If you also have a real resume file (e.g. in data/raw/), test the file path too.
# This proves both code paths work without throwing errors.
test_file_path = "../data/raw/sample_resume.pdf"  # change or leave as-is if file doesn't exist

text_from_file = safe_get_text(row=None, file_path=test_file_path)
if text_from_file:
    print("Loaded from FILE:")
else:
    print(f"No file found at {test_file_path} (or empty) — this is fine, just testing the fallback path.")

print(text_from_file[:500] if text_from_file else "(empty)")

No file found at ../data/raw/sample_resume.pdf (or empty) — this is fine, just testing the fallback path.
(empty)


In [5]:
test_cases = [
    {"row": df.iloc[0], "file_path": None},                          # CSV text only
    {"row": df.iloc[1], "file_path": "../data/raw/does_not_exist.pdf"},  # missing file -> falls back to CSV
    {"row": None, "file_path": "../data/raw/does_not_exist.docx"},   # missing file, no row -> returns ""
]

for i, case in enumerate(test_cases):
    text = safe_get_text(**case)
    print(f"Case {i}: got {len(text)} chars, no error raised. OK.")

Case 0: got 5429 chars, no error raised. OK.
Case 1: got 5560 chars, no error raised. OK.
Case 2: got 0 chars, no error raised. OK.
